# 1. 循环神经的模型结构-计算

- 前提条件
    - 时序数据的训练的要求
        - 金融：股票预测：
        - 自然语言：文本生成
    - 神经网络（多层神经网络）
        - 输入层：input
            - 隐藏层 Hidden
            - ...
            - ...
        - 输出层：output

- 循环神经网络
    - 处理时序数据的核心：记忆功能（增加上下层的矩阵运算）。
        - 每个时序的数据共享权重系数

In [2]:
# 循环神经网络的运算基础
import torch
X1 = torch.randn(3, 1)   # 词向量
W_x = torch.randn(1, 4)  # 输入层的权重矩阵

H0 = torch.randn(3, 4)    # 隐藏层
W_h0 = torch.randn(4, 4)  # 隐藏层权重矩阵

H1 = torch.matmul(X1, W_x) + torch.matmul(H0, W_h0)   # 省略了偏置项 

print(H1)

W_o = torch.randn(4, 1)
O1 = torch.matmul(H1, W_o)

print(O1)


tensor([[ 0.5827,  0.9137,  0.4253, -0.9534],
        [ 0.5105,  2.0337, -0.5246,  0.9165],
        [-0.8152, -1.8275, -1.0093,  0.2086]])
tensor([[-1.5431],
        [ 1.8379],
        [-0.2142]])


In [4]:
# 上面训话神经网络的优化
import torch
X1 = torch.randn(3, 1)   # 词向量
W_x = torch.randn(1, 4)  # 输入层的权重矩阵

H0 = torch.randn(3, 4)    # 隐藏层
W_h0 = torch.randn(4, 4)  # 隐藏层权重矩阵

H1 = torch.matmul(
    torch.cat([X1, H0], dim=1),   # 3 * 5的矩阵
    torch.cat([W_x, W_h0], dim=0)  # 5 * 4的矩阵
)
print(H1)

tensor([[ 2.3442, -1.8519,  1.7691, -1.4304],
        [-2.7598, -0.5264,  2.2322, -4.6496],
        [ 0.3423, -0.8234,  1.9783, -2.2429]])


# 2. 循环神经网络的实现与应用-唐诗

- 循环神经网络的代数式：
    - $ H_t = \tanh ( X_t W_{ih} + b_{ih} + H_{t-1} W_{hh} + b_{hh}) $
    - $ O_t = H_t W_{oh} + b_{oh}$ 

### (1) 模型

In [5]:
import torch
from torch.nn import functional    # flash_attention

In [15]:
class RNN:
    def __init__(self, vocab_size, dim_hidden=64):
        """
            vocab_size: 词袋大小（输入的词向量大小）：输入向量长度
            dim_hidden: 隐藏层的维度大小
            # 前面使用的是列向量，这里使用的是行向量
            torch.randn(size=(self.vocab_size, self.dim_hidden))
        """
        self.vocab_size = vocab_size
        self.dim_hidden = dim_hidden
        # 定义三个权重与偏置项
        self.W_ih = torch.randn(size=(self.vocab_size, self.dim_hidden)) * 0.01  # 采用随机矩阵（没有训练）
        self.W_hh = torch.randn(size=(self.dim_hidden, self.dim_hidden)) * 0.01  # 0.01是防止梯度爆炸 
        self.W_oh = torch.randn(size=(self.dim_hidden, self.vocab_size)) * 0.01  
        self.b_ih = torch.zeros(self.dim_hidden)
        self.b_hh = torch.zeros(self.dim_hidden)
        self.b_oh = torch.zeros(self.vocab_size)

        # 为了能够训练
        self.W_ih.requires_grad = True   # 自动求导
        self.W_hh.requires_grad = True
        self.W_oh.requires_grad = True
        self.b_ih.requires_grad = True
        self.b_hh.requires_grad = True
        self.b_oh.requires_grad = True
        
    def __call__(self, tokens):
        """
            原始输入的shape
            tokens = [batch_size, seq_len]  # "我是中国人" = [12, 345, 56, 78, 987]
            # 向量化的shape
            one_hot = [batch_size, seq_len, vocab_size]  # 转换为词向量或者单热编码
        """
        # 不使用词嵌入，使用单热编码
        # H的初始值。（保存所有的隐藏层数据）
        H = torch.zeros(tokens.shape[-1], self.dim_hidden)
        # 单热编码怎么生成。
        inputs = functional.one_hot(tokens, self.vocab_size).to(torch.float32)  # 转换为小数,shape=（batch_size, seq_len, vocab_size）
        # 计算
        outputs = []  # 输出每个批次的输出
        Hs = []  # 每个批次的隐藏层
        for x in inputs:  # 循环处理每个批次
            H = torch.matmul(x, self.W_ih) + self.b_ih  + torch.matmul(H, self.W_hh) + self.b_hh
            H = torch.tanh(H)  # 激活函数
            y = torch.matmul(H, self.W_oh) + self.b_oh
            outputs.append(y)
            Hs.append(H)
        return torch.cat(outputs, dim=0), torch.cat(Hs, dim=0)

In [19]:
# 测试下
model = RNN(1000, 64)
# 生成一个测试样本
tokens = torch.randint(0, 1000, (64, 24))

outputs, _ = model(tokens)
print(outputs.shape)

torch.Size([1536, 1000])


### (2) 词袋

- 把曹操的赋合并成一个字符串。
- 把字符串按照相等的方式切分成语句（每个语句有固定的长度：seq_len）
- 把多个语句形成批次。
- 比如：整体3000长度
    - 每个句子长度10，切分成300句子
    - 300个句子按照10做成批次：batch_size=10
    - 批次为30批次。
- torchtext不再使用
    - torch官方不在维护。
    - torchtext在python3.13.以后只支持0.6.0本，支持torchtext1.12.0版本必须是python3.12之前的版本
    - 生产级别的Python一般都安装3.12

In [23]:
# 1. 打开文件，读取赋，计数。
import json
from collections import Counter  # python的标准库
data_file = "./datasets/chinese-poetry/曹操诗集/caocao.json"

counter = Counter()   # 一个字就是一个词。
with open(data_file, "r", encoding="utf-8") as fd:
    # 解析成json
    all_poetry = json.load(fd) 
    for poetry in all_poetry:
        contents = "".join(poetry["paragraphs"])   # 取出paragraphs列表，并合并成字符串
        # 统计字频
        for word in contents: # 循环去字符串中的每个
            counter.update(word)  # 自动统计
# print(counter)
# 2. 生成词袋。word2id, id2word, 词袋大小。
words = list(counter.keys())   # 字典
word2id = {word: idx for idx, word in enumerate(words)} 
id2word = {idx: word for idx, word in enumerate(words)}
vocab_size = len(words)  # 词袋大小。
# print(vocab_size)
# print(id2word)

- 图像的特征，还是文本特征都是在任务训练过程中，训练特征。
    - 图像特征：卷积（sobel:一阶导数）
    - 文本特征：词嵌入（Word2Vec算法：skip-Gram, CBOW）。

### (3) 数据集

In [25]:
import json

data_file = "./datasets/chinese-poetry/曹操诗集/caocao.json"

all_data = []
num = 0

with open(data_file, encoding="utf-8") as fd:  # Context Manager
    all_poetry = json.load(fd)
    for poetry in all_poetry:
        num += 1
        contents = "".join(poetry["paragraphs"])
        all_data.append(contents)

text_poetry = "".join(all_data)
# print(num)
# print(text_poetry)

In [29]:
# 编号
text_seq = [word2id[word]  for word in text_poetry]
# print(text_seq)
print(len(text_seq))

3379


In [34]:
# 形成批次集：切分：句，批次
import random
import torch
#   我是中国人
# 我是中国人 
def data_iter_random(corpus_indxcies, batch_size, seq_len):
    """
        corpus_indecies: 编码后的语料。
        batch_size: 批次大小
        seq_len: 时序长度（句子：样本）
    """
    # 计算样本数量
    num_examples = (len(corpus_indxcies)-1) // seq_len 
    # 计算批次数量
    epoch_size = num_examples // batch_size 

    # 随机取样本
    examples_indecies = list(range(num_examples))  # 生成每个样本下标
    random.shuffle(examples_indecies)   # 样本打乱

    def _data(pos):
        return corpus_indxcies[pos: pos + seq_len]
    # 生成批次
    for i in  range(epoch_size):
        i = i * batch_size    # 批次集的开始位置

        batch_indxcies = examples_indecies[i: i +  batch_size]

        x = [_data(j * seq_len)  for j in batch_indxcies]
        y = [_data(j * seq_len + 1) for j in batch_indxcies]   # 标签与样本向后，错一个位置
        yield torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

In [39]:
dataloader = data_iter_random(text_seq, batch_size=8, seq_len=32)
for x, y in dataloader:
    print(x.shape, y.shape)
    # print(x)
    # print(y)
    break

torch.Size([8, 32]) torch.Size([8, 32])


### (4) 单热编码

In [44]:
dataloader = data_iter_random(text_seq, batch_size=8, seq_len=32)
for x, y in dataloader:
    encode = torch.nn.functional.one_hot(x, vocab_size)
    print(encode.shape)
    # print(encode[0,0,:])
    break

torch.Size([8, 32, 982])


### (5) 训练模型

In [45]:
# 数据集
dataloader = data_iter_random(text_seq, batch_size=8, seq_len=32)
# 模型
dim_hidden = 128
model = RNN(vocab_size, dim_hidden)
# 损失函数
loss = torch.nn.CrossEntropyLoss() 
# 优化器（自己更新）

# 超参数轮次，学习率
epoches = 10000
lr = 0.05

In [49]:
for e in range(epoches):
    dataloader = data_iter_random(text_seq, batch_size=8, seq_len=32)  # 每一轮，要重新生成数据集
    total_loss = 0.0
    num = 0 # 批次数（计算平均损失使用）
    for x, y in dataloader:  # 我们自己实现，暂时考虑不了GPU运算
        # 推理
        outputs, _ = model(x) 
        # 计算损失
        # 形状处理一下
        Y = y.contiguous().view(-1)  # 强制改变形状，便于计算损失
        l = loss(outputs, Y.long()) 
        # 自动求导
        l.backward()
        # 梯度更新
        # 为了防止下面的运算影响自动求导
        with torch.no_grad():
            model.W_ih -= model.W_ih.grad * lr
            model.W_hh -= model.W_hh.grad * lr
            model.W_oh -= model.W_oh.grad * lr
            model.b_ih -= model.b_ih.grad * lr
            model.b_hh -= model.b_hh.grad * lr
            model.b_oh -= model.b_oh.grad * lr
            # 梯度清零
            model.W_ih.grad.zero_()
            model.W_hh.grad.zero_()
            model.W_oh.grad.zero_()
            model.b_ih.grad.zero_()
            model.b_hh.grad.zero_()
            model.b_oh.grad.zero_()
            # 统计损失
            total_loss += l.detach().item()
            num += 1
    if e % 100 == 0:
        print(F"批次{e:05d}, 损失:{total_loss / num:.6f}")
print("==========训练完毕==============")

批次00000, 损失:6.885007
批次00100, 损失:5.863479
批次00200, 损失:5.852049
批次00300, 损失:5.825419
批次00400, 损失:5.794721
批次00500, 损失:5.755919
批次00600, 损失:5.711863
批次00700, 损失:5.671349
批次00800, 损失:5.643646
批次00900, 损失:5.581010
批次01000, 损失:5.547908
批次01100, 损失:5.497124
批次01200, 损失:5.453098
批次01300, 损失:5.413486
批次01400, 损失:5.370453
批次01500, 损失:5.323247
批次01600, 损失:5.286419
批次01700, 损失:5.254102
批次01800, 损失:5.207109
批次01900, 损失:5.172610
批次02000, 损失:5.130678
批次02100, 损失:5.092279
批次02200, 损失:5.058277
批次02300, 损失:5.017576
批次02400, 损失:4.983507
批次02500, 损失:4.933680
批次02600, 损失:4.909237
批次02700, 损失:4.858633
批次02800, 损失:4.816665
批次02900, 损失:4.773799
批次03000, 损失:4.723427
批次03100, 损失:4.684401
批次03200, 损失:4.629005
批次03300, 损失:4.583636
批次03400, 损失:4.536181
批次03500, 损失:4.486855
批次03600, 损失:4.428523
批次03700, 损失:4.390278
批次03800, 损失:4.328762
批次03900, 损失:4.276399
批次04000, 损失:4.226215
批次04100, 损失:4.181647
批次04200, 损失:4.120269
批次04300, 损失:4.064917
批次04400, 损失:4.006257
批次04500, 损失:3.959969
批次04600, 损失:3.903706
批次04700, 损失:3

### (6) 推理(生成唐诗)

- 未来可以优化

In [54]:
import torch
prefix = "青"
# 提示词转为编码
tokens = [word2id[word] for word in prefix]  # 广泛适用性 
inputs = torch.tensor(tokens)  # 张量

num_predict = 20  # 循环生成20个字
predicts = []  # 预测结果

for i in range(num_predict):
    outputs, _ = model(inputs)
    # 处理outputs。得到预测结果
    predict = torch.argmax(outputs[0], dim=0).item()

    predicts.append(predict)
    # 使用预测结果作为下轮的预测提示词
    inputs = torch.tensor([predict])


# 使用id2word还原成字
text = "".join(id2word[ids] for ids in predicts)
print(text)


子养有终期。歌以言志，不可追，不可追，不


- $\odot \otimes $
